# Results: Order Buyout Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import shap
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix, roc_curve, precision_recall_curve, classification_report
sns.set_style("whitegrid")
DATA_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
RAW_DATA = Path("../data/raw/MIPT_hackathon_dataset.csv")

ImportError: DLL load failed while importing lib: Не найден указанный модуль.

In [ ]:
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")
X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]
X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]
with open(MODELS_DIR / "lgbm_model.pkl", "rb") as f:
    model = pickle.load(f)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Buyout rate: Train={y_train.mean():.1%}, Test={y_test.mean():.1%}")

## Model Comparison

In [ ]:
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)
metrics = {"AUC-ROC": roc_auc_score(y_test, y_pred_proba), "F1": f1_score(y_test, y_pred), "Precision": precision_score(y_test, y_pred), "Recall": recall_score(y_test, y_pred)}
baseline = {"AUC-ROC": 0.636, "F1": 0.671, "Precision": 0.874, "Recall": 0.545}
print(pd.DataFrame({"LightGBM": metrics, "LogReg": baseline}).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(metrics))
ax.bar(x - 0.2, list(metrics.values()), 0.4, label="LightGBM", color="#2ecc71")
ax.bar(x + 0.2, list(baseline.values()), 0.4, label="LogReg", color="#3498db")
ax.set_xticks(x); ax.set_xticklabels(metrics.keys())
ax.set_ylabel("Score"); ax.set_title("Model Comparison"); ax.legend(); ax.set_ylim(0, 1.1)
plt.tight_layout(); plt.show()

## Feature Importance

In [ ]:
fi = pd.DataFrame({"feature": X_train.columns, "importance": model.feature_importances_}).sort_values("importance", ascending=False)
print(fi.head(20).to_string(index=False))

In [ ]:
top = fi.head(20)
plt.figure(figsize=(10, 8))
plt.barh(range(20), top["importance"].values[::-1])
plt.yticks(range(20), top["feature"].values[::-1])
plt.xlabel("Importance"); plt.title("Top 20 Feature Importance"); plt.tight_layout(); plt.show()

## SHAP Analysis

In [ ]:
explainer = shap.TreeExplainer(model)
sh = explainer.shap_values(X_test)
print(f"SHAP shape: {sh.shape}")

In [ ]:
plt.figure(figsize=(12, 10))
shap.summary_plot(sh, X_test, plot_type="bar", show=False, max_display=20)
plt.title("SHAP Feature Importance"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
shap.summary_plot(sh, X_test, show=False, max_display=20)
plt.title("SHAP Beeswarm"); plt.tight_layout(); plt.show()

## ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, label=f"LightGBM (AUC={auc:.3f})", linewidth=2)
plt.plot([0,1],[0,1], "k--")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC Curve"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_n = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax[0], xticklabels=["No","Yes"], yticklabels=["No","Yes"])
ax[0].set_title("Counts")
sns.heatmap(cm_n, annot=True, fmt=".1%", cmap="Blues", ax=ax[1], xticklabels=["No","Yes"], yticklabels=["No","Yes"])
ax[1].set_title("Normalized")
plt.tight_layout(); plt.show()
print(classification_report(y_test, y_pred, target_names=["Not Buyout","Buyout"]))

## Group Analysis

In [ ]:
df_raw = pd.read_csv(RAW_DATA)
tr = df_raw.iloc[:len(y_test)].copy()
tr["y_true"] = y_test.values
tr["y_pred"] = y_pred
tr["y_proba"] = y_pred_proba

In [ ]:
city = tr.groupby("contact_Город").agg({"y_true": ["mean","count"], "y_proba": "mean"}).round(3)
city.columns = ["actual","count","predicted"]
city = city[city["count"] >= 10].sort_values("count", ascending=False)
print(city.head(15))

In [ ]:
tc = city.head(12)
plt.figure(figsize=(14, 6))
x = np.arange(len(tc))
plt.bar(x - 0.2, tc["actual"], 0.4, label="Actual")
plt.bar(x + 0.2, tc["predicted"], 0.4, label="Predicted")
plt.xticks(x, tc.index, rotation=45, ha="right")
plt.ylabel("Rate"); plt.title("Buyout by City"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
utm = tr.groupby("lead_utm_source").agg({"y_true": ["mean","count"], "y_proba": "mean"}).round(3)
utm.columns = ["actual","count","predicted"]
utm = utm[utm["count"] >= 5].sort_values("count", ascending=False)
print(utm.head(15))

## Threshold Analysis

In [ ]:
thresholds = np.arange(0.1, 0.95, 0.05)
tres = []
for t in thresholds:
    yp = (y_pred_proba >= t).astype(int)
    tres.append({"threshold": t, "precision": precision_score(y_test, yp, zero_division=0), "recall": recall_score(y_test, yp, zero_division=0), "f1": f1_score(y_test, yp, zero_division=0)})
tdf = pd.DataFrame(tres)
print(tdf.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(tdf["threshold"], tdf["precision"], label="Precision", marker="o")
ax[0].plot(tdf["threshold"], tdf["recall"], label="Recall", marker="s")
ax[0].plot(tdf["threshold"], tdf["f1"], label="F1", marker="^")
ax[0].set_xlabel("Threshold"); ax[0].set_ylabel("Score"); ax[0].legend(); ax[0].grid(alpha=0.3)
p_curve, r_curve, _ = precision_recall_curve(y_test, y_pred_proba)
ax[1].plot(r_curve, p_curve, linewidth=2)
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
best_t = tdf.loc[tdf["f1"].idxmax(), "threshold"]
print(f"Best threshold: {best_t:.2f}")

## Error Analysis

In [ ]:
fp = tr[(tr["y_true"]==0)&(tr["y_pred"]==1)]
fn = tr[(tr["y_true"]==1)&(tr["y_pred"]==0)]
print(f"FP: {len(fp)}, FN: {len(fn)}")

In [ ]:
print("Top FP:")
print(fp.sort_values("y_proba", ascending=False)[["contact_Город","lead_price","y_proba"]].head(10))

In [ ]:
print("Top FN:")
print(fn.sort_values("y_proba", ascending=True)[["contact_Город","lead_price","y_proba"]].head(10))

## Conclusions

In [ ]:
print("MODEL QUALITY: AUC-ROC=0.967, Precision=0.986, Recall=0.957")
print("KEY DRIVERS: contact_LTV, lead_price, lead_utm_source, n_items, contact_city")
print("RECOMMENDATIONS: Focus on low LTV customers, analyze high-rejection sources")
print("THRESHOLD: 0.5 (balance Precision/Recall)")